In [1]:
from model import build_model
import torch
from torchvision.transforms import Compose, Resize, CenterCrop, ToTensor, Normalize
try:
    from torchvision.transforms import InterpolationMode
    BICUBIC = InterpolationMode.BICUBIC
except ImportError:
    BICUBIC = Image.BICUBIC
from pprint import pprint
    
def _convert_image_to_rgb(image):
    return image.convert("RGB")

def _transform(n_px):
    return Compose([
        Resize(n_px, interpolation=BICUBIC),
        CenterCrop(n_px),
        _convert_image_to_rgb,
        ToTensor(),
        Normalize((0.48145466, 0.4578275, 0.40821073), (0.26862954, 0.26130258, 0.27577711)),
    ])
model_path = "/share/codes/lzc/ViT-B-16.pt" 
with open(model_path, 'rb') as opened_file:
    state_dict = torch.load(opened_file,map_location='cuda:1').state_dict()
    pprint(state_dict.keys())
    model = build_model(state_dict)
    transform = _transform(model.visual.input_resolution)
    pprint(model)

/opt/conda/envs/mmcv2/lib/python3.11/site-packages/torch/serialization.py:799: UserWarning: 'torch.load' received a zip file that looks like a TorchScript archive dispatching to 'torch.jit.load' (call 'torch.jit.load' directly to silence this warning)
  warnings.warn("'torch.load' received a zip file that looks like a TorchScript archive"


odict_keys(['positional_embedding', 'text_projection', 'logit_scale', 'input_resolution', 'context_length', 'vocab_size', 'visual.class_embedding', 'visual.positional_embedding', 'visual.proj', 'visual.conv1.weight', 'visual.ln_pre.weight', 'visual.ln_pre.bias', 'visual.transformer.resblocks.0.attn.in_proj_weight', 'visual.transformer.resblocks.0.attn.in_proj_bias', 'visual.transformer.resblocks.0.attn.out_proj.weight', 'visual.transformer.resblocks.0.attn.out_proj.bias', 'visual.transformer.resblocks.0.ln_1.weight', 'visual.transformer.resblocks.0.ln_1.bias', 'visual.transformer.resblocks.0.mlp.c_fc.weight', 'visual.transformer.resblocks.0.mlp.c_fc.bias', 'visual.transformer.resblocks.0.mlp.c_proj.weight', 'visual.transformer.resblocks.0.mlp.c_proj.bias', 'visual.transformer.resblocks.0.ln_2.weight', 'visual.transformer.resblocks.0.ln_2.bias', 'visual.transformer.resblocks.1.attn.in_proj_weight', 'visual.transformer.resblocks.1.attn.in_proj_bias', 'visual.transformer.resblocks.1.attn.

In [ ]:
import torch
import torch.nn.functional as F

def contrastive_loss(x, mask, l_shadow, temperature=0.07):
    """
    x: 特征图，形状为 (B, C, H, W) -> (5, 512, 14, 14)
    mask: 阴影掩码，形状为 (B, 1, H, W) -> (5, 1, 14, 14)
    l_shadow: 阴影相关文本描述，形状为 (1, 512)
    temperature: 对比损失的温度系数
    """
    batch_size = x.size(0)
    # 将特征图展平为 (B, H*W, C)
    x_flat = x.view(batch_size, 512, -1).permute(0, 2, 1)  # (5, 196, 512)
    # 将掩码展平为 (B, H*W)
    mask_flat = mask.view(batch_size, -1)  # (5, 196)
    
    # 收集阴影和非阴影区域的特征
    shadow_features = []
    non_shadow_features = []
    for i in range(batch_size):
        current_mask = mask_flat[i]  # (196)
        current_features = x_flat[i]  # (196, 512)
        
        # 提取阴影特征
        shadow_mask = current_mask == 1
        shadow_feat = current_features[shadow_mask]  # (K_i, 512)
        shadow_features.append(shadow_feat)
        
        # 提取非阴影特征
        non_shadow_mask = current_mask == 0
        non_shadow_feat = current_features[non_shadow_mask]  # (196-K_i, 512)
        non_shadow_features.append(non_shadow_feat)
    
    # 拼接所有样本的特征
    x_shadow = torch.cat(shadow_features, dim=0)  # (total_shadow, 512)
    x_non_shadow = torch.cat(non_shadow_features, dim=0)  # (total_non_shadow, 512)
    
    # 处理无阴影或无非阴影的情况
    if x_shadow.size(0) == 0 or x_non_shadow.size(0) == 0:
        return torch.tensor(0.0, device=x.device, requires_grad=True)
    
    # 归一化特征
    l_shadow_norm = F.normalize(l_shadow, p=2, dim=1)  # (1, 512)
    x_shadow_norm = F.normalize(x_shadow, p=2, dim=1)  # (total_shadow, 512)
    x_non_shadow_norm = F.normalize(x_non_shadow, p=2, dim=1)  # (total_non_shadow, 512)
    
    # 计算相似度 (余弦相似度)
    s_pos = torch.mm(l_shadow_norm, x_shadow_norm.T)  # (1, total_shadow)
    s_neg = torch.mm(l_shadow_norm, x_non_shadow_norm.T)  # (1, total_non_shadow)
    
    # 计算对比损失
    sum_neg = torch.exp(s_neg / temperature).sum()
    numerators = torch.exp(s_pos / temperature)  # (1, total_shadow)
    denominators = numerators + sum_neg  # 分母 = 正样本相似度 + 负样本总相似度
    # print(f"sum_neg: {sum_neg}, numerators: {numerators}, denominators: {denominators}")
    
    # 计算每个正样本的损失并取平均
    losses = -torch.log(numerators / (denominators + 1e-8))  # 防止除零
    total_loss = losses.mean()
    
    return total_loss


x = torch.randn(5, 512, 14, 14)
mask = torch.randint(0, 2, (5, 1, 14, 14))
l_shadow = torch.randn(1, 512)
contrastive_loss(x, mask, l_shadow, temperature=0.07)


sum_neg: 591.4708251953125, numerators: tensor([[0.9308, 0.8730, 1.1480, 1.2352, 1.5017, 1.0038, 0.9332, 2.0723, 1.7622,
         0.6780, 1.0499, 0.8353, 1.6826, 1.2432, 2.0807, 1.5644, 0.7432, 1.8624,
         0.7769, 0.4888, 0.9737, 2.5558, 3.1091, 1.4287, 2.2971, 0.5198, 1.8656,
         2.5764, 2.0542, 2.3281, 0.3694, 1.6207, 0.3111, 0.2888, 1.0029, 1.2398,
         0.5915, 2.1805, 0.8903, 1.6513, 2.2712, 1.1911, 1.5684, 1.2840, 2.1425,
         2.2478, 1.2198, 0.5967, 0.7395, 0.8426, 1.6790, 1.1378, 0.8037, 2.8219,
         1.7768, 1.5169, 1.1099, 1.0375, 1.3746, 0.5218, 0.7504, 0.6397, 0.8258,
         2.3343, 1.1450, 1.6101, 1.0227, 0.7526, 1.4160, 2.0021, 1.1404, 0.5775,
         1.2578, 0.3082, 0.7720, 1.5355, 1.2693, 1.5611, 0.7875, 0.8126, 0.5593,
         0.6918, 2.2045, 1.4017, 1.8036, 0.5484, 0.6312, 0.7626, 2.6086, 2.5290,
         0.8351, 0.6080, 2.8758, 0.9089, 0.1832, 0.8299, 1.0204, 0.4939, 0.9389,
         0.7324, 1.4197, 0.9802, 0.9495, 0.6840, 2.4457, 2.6932, 0.49

tensor(6.3205)

In [10]:
clip_x = torch.randn(5, 196, 512)
text_x = torch.randn(1, 512)
a = clip_x @ text_x.transpose(0, 1)
a.shape


torch.Size([5, 196, 1])

In [1]:
import torch
# model = torch.load("/share/codes/lzc/myModels/shadowdiff/saves/Jan09_20.59.48_vl_tt/Jan09_20.59.48_vl_tt_25000.pth")
model = torch.load("/share/codes/lzc/myModels/shadowdiff/saves/Jun05_14.51.34_vl_up2_kernel1_visha_timeAdapter128_20_0_01/Jun05_14.51.34_vl_up2_kernel1_visha_timeAdapter128_20_0_01_25000.pth")

for key in model.keys():
    if "scale" in key:
        print(f"{key}:{model[key]}")
    # print(key)

backbone.adapters.0.shadow_scale:0.003370598889887333
backbone.adapters.0.bg_scale:0.0011077066883444786
backbone.adapters.1.shadow_scale:0.0017618497367948294
backbone.adapters.1.bg_scale:0.003866607090458274
backbone.adapters.2.shadow_scale:0.0018092090031132102
backbone.adapters.2.bg_scale:0.005839247722178698
backbone.adapters.3.shadow_scale:-0.002261766232550144
backbone.adapters.3.bg_scale:-0.001302358927205205
backbone.temporal_adapters.0.scale:0.01137525774538517
backbone.temporal_adapters.1.scale:0.0007496649050153792
backbone.temporal_adapters.2.scale:0.004030507057905197
backbone.temporal_adapters.3.scale:0.004811954218894243
backbone.temporal_adapters.4.scale:0.006412430200725794
backbone.temporal_adapters.5.scale:0.006872689351439476
backbone.temporal_adapters.6.scale:0.009442996233701706
backbone.temporal_adapters.7.scale:0.0062226830050349236
backbone.temporal_adapters.8.scale:0.010963509790599346
backbone.temporal_adapters.9.scale:0.009816423989832401
backbone.temporal_